In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
cd /content/drive/MyDrive/mls-asr-main/hw1-asr/glm_asr_triton_template/

/content/drive/MyDrive/mls-asr-main/hw1-asr/glm_asr_triton_template


In [4]:
rm -rf ~/.triton/cache/

In [7]:
import shutil, os

# Clear Triton's compiled kernel cache
cache_dir = os.path.expanduser("~/.triton/cache")
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print("Triton cache cleared!")
else:
    print("No cache found.")

Triton cache cleared!


In [8]:
# Run this in Colab to check there's no break in your Drive file
with open("/content/drive/MyDrive/mls-asr-main/hw1-asr/glm_asr_triton_template/attention.py") as f:
    content = f.read()

if "break" in content:
    print("WARNING: old version with break still on Drive!")
else:
    print("File is clean - no break statement found.")

In [9]:
with open("/content/drive/MyDrive/mls-asr-main/hw1-asr/glm_asr_triton_template/attention.py") as f:
    content = f.read()

# Remove the early-exit causal break block
old_code = """        if IS_CAUSAL:
            # Mask out key blocks that are entirely after the query block
            if start_k > (pid_q + 1) * BLOCK_Q - 1:
                break"""

new_code = """        # IS_CAUSAL masking is handled below with tl.where"""

if old_code in content:
    content = content.replace(old_code, new_code)
    with open("/content/drive/MyDrive/mls-asr-main/hw1-asr/glm_asr_triton_template/attention.py", "w") as f:
        f.write(content)
    print("File patched successfully!")
else:
    print("Pattern not found - printing the loop section so we can find the exact text:")
    # Find and print the relevant section
    idx = content.find("for start_k in range")
    print(content[idx:idx+400])

File patched successfully!


In [10]:
import shutil, os
shutil.rmtree(os.path.expanduser("~/.triton/cache"), ignore_errors=True)
print("Cache cleared - ready to run!")

Cache cleared - ready to run!


In [5]:
!python attention.py

Testing Triton Attention...

Basic attention:
  Output shape: torch.Size([2, 4, 16, 64])

Causal attention:
  Output shape: torch.Size([2, 4, 16, 64])

With attention mask:
  Output shape: torch.Size([2, 4, 16, 64])

Grouped Query Attention (GQA):
  Output shape: torch.Size([2, 4, 16, 64])

Output statistics:
  Mean: -0.0095
  Std:  0.3554
  Min:  -1.5437
  Max:  1.5758

Triton Attention working!


In [12]:
!python layers.py 

Testing Triton Layers...

=== RMSNorm ===
Input: torch.Size([2, 16, 256]) -> Output: torch.Size([2, 16, 256])

=== LayerNorm ===
Input: torch.Size([2, 16, 256]) -> Output: torch.Size([2, 16, 256])

=== GELU ===
Input: torch.Size([2, 16, 256]) -> Output: torch.Size([2, 16, 256])

=== SiLU ===
Input: torch.Size([2, 16, 256]) -> Output: torch.Size([2, 16, 256])

=== Linear ===
Input: torch.Size([2, 16, 256]) -> Output: torch.Size([2, 16, 512])

=== Embedding ===
Input: torch.Size([2, 16]) -> Output: torch.Size([2, 16, 256])

=== Softmax ===
Input: torch.Size([2, 4, 16, 16]) -> Output: torch.Size([2, 4, 16, 16])
Sum along last axis: 1.000000 (should be 1.0)

=== MLP ===
Input: torch.Size([2, 16, 256]) -> Output: torch.Size([2, 16, 256])

=== RMSNorm + Linear (Fused) ===
Input: torch.Size([16, 256]) -> Output: torch.Size([16, 512])
Max diff: 0.000038

All Triton layers working!


In [13]:
!python rope.py 

Testing Triton RoPE...
Cos shape: torch.Size([16, 64])
Sin shape: torch.Size([16, 64])
Q rotated shape: torch.Size([2, 4, 16, 64])
K rotated shape: torch.Size([2, 4, 16, 64])

Testing partial RoPE (50%):
Q rotated (partial) shape: torch.Size([2, 4, 16, 64])

Triton RoPE working!


In [22]:
import shutil, os

# Mount Drive if not already mounted
from google.colab import drive
drive.mount('/content/drive')

# Copy your local working files to Drive
src = "/Users/sans/Downloads/mls-asr-main"  # where your updated files are
dst = "/content/drive/MyDrive/mls-asr-main"

# Copy all files
for filename in os.listdir(src):
    src_file = os.path.join(src, filename)
    dst_file = os.path.join(dst, filename)
    if os.path.isfile(src_file):
        shutil.copy2(src_file, dst_file)
        print(f"Copied: {filename}")

print("Done!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: [Errno 2] No such file or directory: '/Users/sans/Downloads/mls-asr-main'